# Milestone 4 lab, notebook 1 of 8: A rule lives in dbt

## Where we are

| Notebook | What you do |
|---|---|
| **1. A rule lives in dbt** (you are here) | Move a check out of a notebook and into a file that dbt runs |
| 2. How serious is it | Tell dbt which failures stop the night. Keep the rows a test returns |
| 3. Test the SQL, not the data | Check your own query on four rows typed by hand |
| 4. A gate, and where it stands | Run one rule in three places, and see what each place can see |
| 5. Warn, block or set aside | Decide what a failure does. Then block a real run in Airflow |
| 6. Quarantine | Set the bad rows aside with a reason, and let the good rows go on |
| 7. Test the tests | Plant a fault, and prove its gate catches it. One check in four tools |
| 8. The production version | Run the whole week through Airflow. See what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import (
    after_the_gates, before_the_clean_gate, dbt, prepare_week, show_file, sql,
)

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August, through three gates.
Sunday night ran 47 tests. 1 of 6 stores can be paid.


## Step 1: Where you stopped

**Lotus Retails** is a chain of six supermarkets. Customers pay at a **till**: the machine at the counter where a customer pays. It is also called a cash register.

At the end of milestone 3 you had 37 checks on 7 tables. Each had a reason and a proof. And not one of them was in the pipeline. On Monday, **Tomas Reyes**, the manager of store `S03`, would still have been paid a bonus on $796.18 for a day when his tills took $777.20.

A check in a notebook protects nobody. In this milestone the checks move into the pipeline. That takes two tools, and beginners mix them up. So this box is on every page:

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, and what happens when it fails.

This notebook is about the first half: the rule, in dbt.

## Step 2: What dbt is, in three sentences

**dbt** is a tool that keeps SQL in files and runs it against the warehouse. Most teams use it to build tables. Here it does one job only: it runs tests.

A dbt **test** is what you have called a check since milestone 1: a query that returns the rows that break a rule. No rows means the test passes.

The dbt project is the folder `dbt_project/` in this milestone. You never start dbt alone: the pipeline starts it. For the lab, `dbt(...)` runs one dbt command and prints what dbt says.

## Step 3: A generic test. One line switches it on

Remember the five standard checks of milestone 1? Three of them are so common that dbt ships them ready-made. You do not write their SQL. You name a column and switch the test on.

Here they are, in the file that lists the tables.

In [2]:
show_file("dbt_project/models/sources.yml", "# Three of the five standard checks", 20)

# dbt_project/models/sources.yml, from line 65
        # Three of the five standard checks from milestone 1. In dbt they are GENERIC tests:
        # ready-made, and switched on with one line each.
        columns:
          - name: line_id
            data_tests:
              - not_null:
                  name: standard_key_is_not_null
                  config: {tags: [clean], severity: error,
                           meta: {number: 102, action: block, lens: data, question: "Complete?", protects: ""}}
              - unique:
                  name: standard_key_is_unique
                  config: {tags: [clean], severity: error,
                           meta: {number: 103, action: block, lens: data, question: "Unique?", protects: ""}}
          - name: line_type
            data_tests:
              - accepted_values:
                  name: standard_status_is_allowed
                  arguments:
                    values: ["sale", "refund"]
                  config: {tags: [cle

Read past the `config:` lines for now. The tests themselves are three words: `not_null`, `unique`, `accepted_values`.

A ready-made test like this is called a **generic test**. It is written once, and it works on any column of any table.

Before you run them, one thing. The nightly run has already been through the gates, and it set the bad rows aside. To see the tests work, put the clean tables back as they are **before** any gate looks at them.

In [3]:
before_the_clean_gate()

The clean tables are rebuilt from the raw rows. No gate has looked at them yet.


In [4]:
dbt("test", "--select", "standard_key_is_not_null", "standard_key_is_unique", "standard_status_is_allowed")

Found 4 models, 46 data tests, 25 sources, 480 macros, 2 unit tests
1 of 3 PASS standard_key_is_not_null  ...  [PASS in 0.04s]
2 of 3 PASS standard_key_is_unique  ...  [PASS in 0.02s]
3 of 3 PASS standard_status_is_allowed  ...  [PASS in 0.02s]
Done. PASS=3 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=3


Three tests, three `PASS`. They are the same three green checks as in milestone 1, and they are as blind as they were then: the refund with the wrong sign is in that table right now.

## Step 4: A singular test. Your own query, in a file

In milestone 2 you wrote this rule by hand: a refund gives money back. Here is that query again.

In [5]:
sql("""
    SELECT line_id, store_id, sales_date, line_type, quantity, amount_cents
    FROM clean.sales_lines
    WHERE (line_type = 'refund' AND (amount_cents >= 0 OR quantity >= 0))
       OR (line_type = 'sale' AND (amount_cents <= 0 OR quantity <= 0))
""")

,line_id,store_id,sales_date,line_type,quantity,amount_cents
0,S03-0824-000008,S03,2026-08-24,refund,-1,949


One row: the refund of $9.49 at `S03`. Now look at the same rule as a dbt test. It is one file.

In [6]:
show_file("dbt_project/tests/clean/refund_gives_money_back.sql")

# dbt_project/tests/clean/refund_gives_money_back.sql
-- A refund gives money back
--
-- Prepare first: Read sales and refunds apart
-- The limit:     A refund has a quantity and an amount below zero. A sale has both above zero
-- When it fails: QUARANTINE: the rows this test returns are set aside with this reason. The good rows carry on.
{{ config(tags=['clean'], severity='warn', meta={'number': 6, 'action': 'quarantine', 'lens': 'business', 'question': 'Valid?', 'protects': 'Store managers and the head of retail operations: net sales, so pay', 'sets_aside': 'sales_lines'}) }}

SELECT line_id, store_id, sales_date, line_type, quantity, amount_cents
FROM {{ source('clean', 'sales_lines') }}
WHERE (line_type = 'refund' AND (amount_cents >= 0 OR quantity >= 0))
   OR (line_type = 'sale' AND (amount_cents <= 0 OR quantity <= 0))


It is your query. Three things were added around it.

- **Comments on top.** The rule in words, what must be prepared, the limit, and what happens when it fails. The check card, on the test itself.
- **`{{ config(...) }}`.** Notes for dbt and for the pipeline. Notebook 2 and notebook 5 read them.
- **`{{ source('clean', 'sales_lines') }}`** in place of `clean.sales_lines`. It is how dbt names a table that it did not build. dbt turns it back into the table's name.

One query in one file, written for one rule: that is a **singular test**.

Run it.

In [7]:
dbt("test", "--select", "refund_gives_money_back")

Found 4 models, 46 data tests, 25 sources, 480 macros, 2 unit tests
1 of 1 WARN 1 refund_gives_money_back  ...  [WARN 1 in 0.04s]
Done. PASS=0 WARN=1 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


`WARN 1`. The test returned one row, the same row your query found. Why it says `WARN` and not `FAIL` is the next notebook.

## Step 5: Your own generic test

Sometimes you write a query, and then need it again on another table. "This combination of columns is there once" is such a rule: you needed it for the tills, for stock counts, for supplier prices.

You can write a generic test of your own, once.

In [8]:
show_file("dbt_project/macros/test_unique_together.sql")

# dbt_project/macros/test_unique_together.sql
{#- A generic test you write once and use on any table:

        data_tests:
          - unique_together:
              arguments:
                columns: [store_id, till_id, business_date]

    It returns every combination of the columns that is there more than once. -#}
{% test unique_together(model, columns) %}

select {{ columns | join(', ') }}, count(*) as times
from {{ model }}
group by {{ columns | join(', ') }}
having count(*) > 1

{% endtest %}


`model` and `columns` are blanks. Each use fills them in. Here it is used on `till_totals`, with three columns.

In [9]:
show_file("dbt_project/models/sources.yml", "# A generic test with more than one column", 6)

# dbt_project/models/sources.yml, from line 53
        # A generic test with more than one column: written once, in macros/, used here.
        data_tests:
          - unique_together:
              name: one_total_per_till_per_day
              arguments:
                columns: [store_id, till_id, business_date]


In [10]:
dbt("test", "--select", "one_total_per_till_per_day")

Found 4 models, 46 data tests, 25 sources, 480 macros, 2 unit tests
1 of 1 PASS one_total_per_till_per_day  ...  [PASS in 0.04s]
Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


| | Generic test | Singular test |
|---|---|---|
| What it is | A rule with blanks, written once | One query, for one rule |
| Where it lives | `macros/`, or inside dbt | One file in `tests/` |
| How you use it | One line in the list of tables | The file is the test |
| Good for | Not empty, unique, allowed values | A rule of the business, a join, a comparison |

Most of your 37 checks are singular tests. A rule with a reason is rarely a rule with blanks.

## Step 6: All the tests of one place

Every test has a tag: `landing`, `clean` or `publish`. It says where in the pipeline the test runs. Run every test tagged `clean`, and list only the ones that found something.

In [11]:
dbt("test", "--select", "tag:clean", only_failed=True)

Found 4 models, 46 data tests, 25 sources, 480 macros, 2 unit tests
1 of 39 WARN 1 a_count_for_every_day  ...  [WARN 1 in 0.25s]
2 of 39 WARN 1 believable_quantity  ...  [WARN 1 in 0.02s]
8 of 39 WARN 3 every_sale_has_a_product  ...  [WARN 3 in 0.03s]
11 of 39 WARN 4 inside_opening_hours  ...  [WARN 4 in 0.03s]
12 of 39 WARN 1 line_adds_up  ...  [WARN 1 in 0.02s]
16 of 39 WARN 1 normal_volume  ...  [WARN 1 in 0.03s]
18 of 39 WARN 1 one_person_once  ...  [WARN 1 in 0.02s]
22 of 39 WARN 6 price_of_the_day  ...  [WARN 6 in 0.02s]
24 of 39 WARN 1 refund_gives_money_back  ...  [WARN 1 in 0.02s]
39 of 39 WARN 171 true_key_unique  ...  [WARN 171 in 0.04s]
(29 tests passed, and are not listed)
Done. PASS=29 WARN=10 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=39


Count the lines: ten tests found bad rows, and twenty-nine passed. The refund, the receipt sent twice, the bananas, the soup, the butter, the member who signed up twice: each is in that list, found by a file in `dbt_project/tests/clean/`.

And still nothing happened. dbt found the rows and stopped there. dbt holds the rule. It does not decide what happens next. That is the second half, and it starts in notebook 2.

Put the warehouse back as the nightly run left it.

In [12]:
after_the_gates()

Sunday's night ran again. The warehouse is as the nightly run left it.


## What you now know

- dbt keeps SQL in files and runs it. In this project it only runs tests.
- A **generic test** is ready-made and switched on with one line, such as `not_null` or `unique`.
- A **singular test** is one query in one file, written for one rule.
- Your checks did not change. They moved from a notebook into files, with their reason written on top.
- dbt holds the rule. It does not act on the result.

**Say it in an interview:** "We kept every quality rule as a dbt test. The common ones were generic tests, one line each. The rules of the business were singular tests: one query per file, with the reason and the limit written on top. dbt held the rules, and the scheduler decided what a failure did."

**The theory:** [dbt Tests: Generic, Singular & Custom](https://www.d8loop.com/modules/knowledge-corner/dbt/dbt-tests-generic-singular-and-custom), and [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "Checks as dbt tests".

Next: [notebook 2](02_how_serious_is_it.ipynb). The test said `WARN`. Who decided that?